# 1장. 훈련 오차와 시험 오차

지난 일을 잘 설명하는 식이 다가올 일도 잘 맞히는가.

**준비:** 한국전력거래소 「시간별 전국 전력수요량」 CSV 세 개(2023·2024·2025년)와 기상청 종관기상관측(ASOS) 시간자료 CSV 두 개(2023·2025년)를 왼쪽 파일 창에 올린 뒤 위에서부터 차례로 실행한다. 『데이터 분석 완전 정복』 5장에서 쓴 파일 그대로다.

## 1.1 훈련 자료를 오차 없이 맞추는 방법

In [ ]:
import glob
import numpy as np
import pandas as pd

pd.set_option("display.unicode.east_asian_width", True)

def first_line(path):
    with open(path, encoding="cp949") as f:
        return f.readline()

csvs = sorted(glob.glob("*.csv"))
power_files = [f for f in csvs if first_line(f).startswith("날짜,1시")]
weather_files = [f for f in csvs if first_line(f).startswith("지점,지점명")]

wide = pd.concat([pd.read_csv(f, encoding="cp949") for f in power_files]).dropna(subset=["날짜"])
long = wide.melt("날짜", var_name="시", value_name="MWh")
long["시각"] = pd.to_datetime(long["날짜"]) + pd.to_timedelta(long["시"].str.replace("시", "").astype(int), unit="h")
power = long.set_index("시각")["MWh"].sort_index() / 1000          # GW

FIVE = ["서울", "대전", "대구", "부산", "광주"]
w = pd.concat([pd.read_csv(f, encoding="cp949", usecols=["지점명", "일시", "기온(°C)"]) for f in weather_files])
w["일시"] = pd.to_datetime(w["일시"])
temp = w[w["지점명"].isin(FIVE)].pivot(index="일시", columns="지점명", values="기온(°C)").interpolate()

h = pd.DataFrame({"GW": power, "기온": temp.mean(axis=1)}).dropna()
h["날짜"] = (h.index - pd.Timedelta(hours=1)).normalize()
d = h.groupby("날짜").agg(GW=("GW", "mean"), 기온=("기온", "mean"))
print(len(power_files), len(weather_files), len(h), len(d))

In [ ]:
HOLIDAYS = pd.to_datetime([
    "2023-01-01", "2023-01-21", "2023-01-22", "2023-01-23", "2023-01-24", "2023-03-01",
    "2023-05-05", "2023-05-27", "2023-05-29", "2023-06-06", "2023-08-15", "2023-09-28",
    "2023-09-29", "2023-09-30", "2023-10-02", "2023-10-03", "2023-10-09", "2023-12-25",
    "2025-01-01", "2025-01-27", "2025-01-28", "2025-01-29", "2025-01-30", "2025-03-01",
    "2025-03-03", "2025-05-05", "2025-05-06", "2025-06-03", "2025-06-06", "2025-08-15",
    "2025-10-03", "2025-10-05", "2025-10-06", "2025-10-07", "2025-10-08", "2025-10-09",
    "2025-12-25"])

weekday = d[(d.index.dayofweek < 5) & ~d.index.isin(HOLIDAYS)]
train = weekday[weekday.index.year == 2023]
test = weekday[weekday.index.year == 2025]

print(len(train), len(test))
print(train.head(3).round(2).to_string())

In [ ]:
def rmse(actual, predicted):
    """오차를 제곱해 평균하고 제곱근을 씌운다. 단위는 GW 그대로다."""
    miss = np.asarray(actual, dtype=float) - np.asarray(predicted, dtype=float)
    return round(float(np.sqrt((miss ** 2).mean())), 2)

always = train["GW"].mean()
print(round(float(always), 2), rmse(train["GW"], always), rmse(test["GW"], always))

In [ ]:
def neighbors(temps, k, known):
    """기온이 가장 가까운 k일을 known에서 찾아 그날들의 수요를 평균한다."""
    T, Y = known["기온"].to_numpy(), known["GW"].to_numpy()
    return np.array([Y[np.argsort(np.abs(T - t))[:k]].mean() for t in np.asarray(temps, dtype=float)])

print(rmse(train["GW"], neighbors(train["기온"], 1, train)))

In [ ]:
t = 2.21                                  # 2025년 1월 31일의 평균 기온
near = train.assign(거리=(train["기온"] - t).abs()).sort_values("거리").head(5)
print(near.round(2).to_string())
print([round(float(near["GW"].head(k).mean()), 2) for k in (1, 3, 5)])

In [ ]:
print(rmse(test["GW"], neighbors(test["기온"], 1, train)))

In [ ]:
T = train["기온"].to_numpy()
twin = [train.index[np.argmin(np.abs(T - t))] for t in test["기온"]]
look = test.assign(닮은날=[t.date() for t in twin], 그날기온=train.loc[twin, "기온"].to_numpy(),
                   답=train.loc[twin, "GW"].to_numpy())
look["오차"] = look["GW"] - look["답"]
print(look.sort_values("오차").round(2).head(4).to_string())
print(look["닮은날"].value_counts().head(3).to_string())

In [ ]:
lo, hi = train["기온"].min(), train["기온"].max()
print(round(float(lo), 2), round(float(hi), 2), round(float(test["기온"].min()), 2), round(float(test["기온"].max()), 2))
print(int((test["기온"] < lo).sum()), int((test["기온"] > hi).sum()))

## 1.2 이웃의 수와 과적합·과소적합

In [ ]:
KS = [1, 2, 3, 5, 8, 12, 20, 30, 50, 100, 247]
rows = [(k, rmse(train["GW"], neighbors(train["기온"], k, train)),
            rmse(test["GW"], neighbors(test["기온"], k, train))) for k in KS]
sweep = pd.DataFrame(rows, columns=["이웃", "2023년", "2025년"]).set_index("이웃")
print(sweep.to_string())

In [ ]:
!pip -q install koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)


x = np.linspace(-9, 30.5, 400)

In [ ]:
# 그림 1-3
fig, axes = plt.subplots(3, 1, figsize=(7, 8.2), sharex=True)
notes = {1: "모든 점을 지난다 (과적합)", 12: "점들 사이를 지나며 U자 모양만 남긴다", 100: "15도 부근의 낮은 구간이 사라진다 (과소적합)"}
for ax, k in zip(axes, [1, 12, 100]):
    ax.scatter(train["기온"], train["GW"], s=9, color=to_rgba(NAVY, 0.25), edgecolor=NAVY, lw=0.3)
    ax.plot(x, neighbors(x, k, train), color=BRICK, lw=1.3)
    frame(ax, f"이웃 {k}일: {notes[k]}")
    ax.set_ylabel("하루 평균 전력수요(GW)", fontsize=8)
    ax.tick_params(labelbottom=True)
    ax.text(30.3, 56.5, f"유효 파라미터 수 약 {len(train) / k:.0f}개" if k < 100 else f"유효 파라미터 수 약 {len(train) / k:.1f}개", fontsize=8, color=TEAL, ha="right")
axes[-1].set_xlabel("하루 평균 기온(°C)", fontsize=8)
fig.tight_layout()
fig.savefig("fig_1_3.png", dpi=300)
plt.show()

In [ ]:
# 그림 1-4
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(sweep.index, sweep["2023년"], marker="o", ms=4, color=GRAY, lw=1.6, label="2023년: 훈련 오차")
ax.plot(sweep.index, sweep["2025년"], marker="o", ms=4, color=NAVY, lw=1.6, label="2025년: 시험 오차")
best = sweep["2025년"].idxmin()
ax.axvline(best, color=TEAL, lw=1.0, ls="--")
ax.text(best * 1.08, 5.2, f"시험 오차가 가장 작은 곳\n이웃 {best}일, {sweep.loc[best, '2025년']:.2f}GW", color=TEAL, fontsize=8, va="top")
ax.annotate("과적합", (1.15, 2.2), fontsize=8.5, color=BRICK, weight="bold")
ax.annotate("과소적합", (95, 2.2), fontsize=8.5, color=BRICK, weight="bold")
for k in [1, 100, 247]:
    ax.text(k, sweep.loc[k, "2025년"] + 0.18, f"{sweep.loc[k, '2025년']:.2f}", ha="center", fontsize=7.5, color=NAVY)
ax.set_xscale("log")
ax.set_xticks(KS)
ax.set_xticklabels(KS)
ax.minorticks_off()
ax.set_ylim(-0.3, 7.2)
ax.set_xlabel("답을 낼 때 평균하는 이웃의 수 k", fontsize=8)
ax.set_ylabel("RMSE(GW)", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="upper left")
frame(ax)
fig.tight_layout()
fig.savefig("fig_1_4.png", dpi=300)
plt.show()

## 1.3 오차를 편향과 분산으로 나누기

In [ ]:
def spread(k, rounds=200, seed=0):
    """훈련 자료를 다시 뽑아 식을 rounds번 만들고, 2025년의 날마다 답이 얼마나 달라지는지 잰다."""
    rng = np.random.default_rng(seed)
    y = test["GW"].to_numpy()
    answers = np.array([
        neighbors(test["기온"], k, train.iloc[rng.integers(0, len(train), len(train))])
        for _ in range(rounds)
    ])
    center = answers.mean(axis=0)                 # 날마다, 200개 식이 낸 답의 평균
    off = ((y - center) ** 2).mean()              # 평균 답이 실제와 어긋난 크기
    var = answers.var(axis=0).mean()              # 식마다 답이 달라지는 크기
    total = ((y - answers) ** 2).mean()           # 200개 식의 제곱 오차를 모두 평균한 값
    return off, var, total

rows = [(k, *spread(k)) for k in KS]
bv = pd.DataFrame(rows, columns=["이웃", "평균 답의 오차", "분산", "제곱 오차"]).set_index("이웃")
bv["합"] = bv["평균 답의 오차"] + bv["분산"]
print(bv[["평균 답의 오차", "분산", "합", "제곱 오차"]].round(2).to_string())

In [ ]:
noise = float(((train["GW"] - neighbors(train["기온"], 12, train)) ** 2).mean())
check = pd.DataFrame({"잰 분산": bv["분산"], "잡음 ÷ k": [noise / k for k in KS]}).round(2)
print(round(noise, 2))
print(check.to_string())

In [ ]:
# 그림 1-5
fig, ax = plt.subplots(figsize=(7, 3.8))
pos = np.arange(len(KS))
ax.bar(pos, bv["평균 답의 오차"], width=0.62, color=to_rgba(NAVY, 0.22), edgecolor=NAVY, lw=0.9, label="평균 답의 오차 (편향 제곱 + 잡음)")
ax.bar(pos, bv["분산"], bottom=bv["평균 답의 오차"], width=0.62, color=to_rgba(BRICK, 0.55), edgecolor=BRICK, lw=0.9, label="분산")
for i, k in enumerate(KS):
    ax.text(i, bv.loc[k, "합"] + 0.6, f"{bv.loc[k, '합']:.1f}", ha="center", fontsize=7.5, color=NAVY)
ax.annotate("분산이 오차를 키우는 구간\n(식이 훈련 자료의 우연을 따라간다)", (0.35, 17.5), (1.6, 27), fontsize=8, color=BRICK, arrowprops={"arrowstyle": "->", "color": BRICK, "lw": 0.8})
ax.annotate("편향이 오차를 키우는 구간\n(식이 U자 형태를 따라가지 못한다)", (9.6, 23.5), (5.4, 34), fontsize=8, color=NAVY, arrowprops={"arrowstyle": "->", "color": NAVY, "lw": 0.8})
low = int(np.argmin(bv["합"].to_numpy()))
ax.annotate("합이 가장 작은 곳", (low, bv["합"].iloc[low] + 2.0), (low, 19.5), fontsize=8, color=TEAL, ha="center", arrowprops={"arrowstyle": "->", "color": TEAL, "lw": 0.8})
ax.set_xticks(pos, KS)
ax.set_xlabel("이웃의 수 k", fontsize=8)
ax.set_ylabel("제곱 오차의 평균(GW 제곱)", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="upper left")
ax.set_ylim(0, 47)
frame(ax)
fig.tight_layout()
fig.savefig("fig_1_5.png", dpi=300)
plt.show()

## 1.4 식의 복잡도와 자료의 양

In [ ]:
def polynomial(degree, known):
    """known에 degree차 다항식을 맞추고, 기온을 넣으면 수요를 돌려주는 함수를 만든다."""
    mu, sd = known["기온"].mean(), known["기온"].std()
    def X(temps):
        return np.vander((np.asarray(temps, dtype=float) - mu) / sd, degree + 1)
    coef = np.linalg.lstsq(X(known["기온"]), known["GW"].to_numpy(), rcond=None)[0]
    return lambda temps: X(temps) @ coef

DEGREES = [1, 2, 3, 4, 6, 8, 10, 12]
rows = []
for g in DEGREES:
    f = polynomial(g, train)
    rows.append((g, rmse(train["GW"], f(train["기온"])), rmse(test["GW"], f(test["기온"]))))
print(pd.DataFrame(rows, columns=["차수", "2023년", "2025년"]).set_index("차수").to_string())

In [ ]:
raw = np.vander(train["기온"].to_numpy(), 13)
scaled = np.vander(((train["기온"] - train["기온"].mean()) / train["기온"].std()).to_numpy(), 13)
print(f"{np.linalg.cond(raw):.1e}", f"{np.linalg.cond(scaled):.1e}")

In [ ]:
few = train.sample(30, random_state=1).sort_index()
rows = []
for g in DEGREES:
    f = polynomial(g, few)
    rows.append((g, rmse(few["GW"], f(few["기온"])), rmse(test["GW"], f(test["기온"]))))
print(round(float(few["기온"].min()), 1), round(float(few["기온"].max()), 1))
print(pd.DataFrame(rows, columns=["차수", "30일", "2025년"]).set_index("차수").to_string())

In [ ]:
# 그림 1-6
fig, ax = plt.subplots(figsize=(7, 3.9))
ax.axvspan(few["기온"].min(), few["기온"].max(), color=TEAL, alpha=0.08)
ax.text(few["기온"].mean(), 46.3, "30일의 기온이 있는 구간", color=TEAL, fontsize=8, ha="center")
ax.scatter(train["기온"], train["GW"], s=8, color=to_rgba(GRAY, 0.35), edgecolor=GRAY, lw=0.3, label="2023년 평일 247일")
ax.scatter(few["기온"], few["GW"], s=20, color=NAVY, lw=0, label="그 가운데 뽑은 30일")
ax.plot(x, polynomial(3, few)(x), color=NAVY, lw=1.4, label="30일로 추정한 3차식")
ax.plot(x, polynomial(8, few)(x), color=BRICK, lw=1.4, label="30일로 추정한 8차식")
ax.annotate("8차식: 범위 밖에서\n답이 급격히\n작아진다", (-1.35, 60), (-9.4, 48.5), fontsize=8, color=BRICK, bbox={"fc": "white", "ec": "none", "alpha": 0.85, "pad": 1.5}, arrowprops={"arrowstyle": "->", "color": BRICK, "lw": 0.8})
ax.annotate("3차식: 범위 밖에서도\n완만하게 변한다", (-6, float(polynomial(3, few)([-6])[0])), (-8.6, 90.5), fontsize=8, color=NAVY, arrowprops={"arrowstyle": "->", "color": NAVY, "lw": 0.8})
ax.set_ylim(45, 97)
ax.set_xlabel("하루 평균 기온(°C)", fontsize=8)
ax.set_ylabel("하루 평균 전력수요(GW)", fontsize=8)
ax.legend(frameon=False, fontsize=8, ncol=2, loc="upper center")
frame(ax)
fig.tight_layout()
fig.savefig("fig_1_6.png", dpi=300)
plt.show()

In [ ]:
def hinge(known):
    """꺾이는 두 온도를 0.5도 간격으로 찾아, 그보다 추운 만큼과 더운 만큼에 기울기를 하나씩 준다."""
    def X(temps, cold_at, hot_at):
        t = np.asarray(temps, dtype=float)
        return np.column_stack([np.ones(len(t)), np.maximum(cold_at - t, 0), np.maximum(t - hot_at, 0)])
    y, best = known["GW"].to_numpy(), None
    for cold_at in np.arange(8, 20.5, 0.5):
        for hot_at in np.arange(cold_at, 26.5, 0.5):
            coef = np.linalg.lstsq(X(known["기온"], cold_at, hot_at), y, rcond=None)[0]
            sse = float(((y - X(known["기온"], cold_at, hot_at) @ coef) ** 2).sum())
            if best is None or sse < best[0]:
                best = (sse, cold_at, hot_at, coef)
    _, cold_at, hot_at, coef = best
    f = lambda temps: X(temps, cold_at, hot_at) @ coef
    f.knots, f.coef = (float(cold_at), float(hot_at)), coef     # 꺾이는 온도와 계수를 함께 보관한다
    return f

bent = hinge(train)
print(rmse(train["GW"], bent(train["기온"])), rmse(test["GW"], bent(test["기온"])))
bent30 = hinge(few)
print(rmse(few["GW"], bent30(few["기온"])), rmse(test["GW"], bent30(test["기온"])))

In [ ]:
print(bent.knots, np.round(bent.coef, 2))
print(bent30.knots, np.round(bent30.coef, 2))

In [ ]:
# 그림 1-7
c1, c2 = bent.knots
b0, b1, b2 = bent.coef
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.scatter(train["기온"], train["GW"], s=9, color=to_rgba(NAVY, 0.22), edgecolor=NAVY, lw=0.3, label="2023년 평일(훈련 자료)")
ax.scatter(test["기온"], test["GW"], s=9, color=to_rgba(BRICK, 0.30), edgecolor=BRICK, lw=0.3, label="2025년 평일(시험 자료)")
ax.plot(x, bent(x), color=TEAL, lw=2.0, label="2023년으로 추정한 꺾은 직선")
for c in (c1, c2):
    ax.axvline(c, color=TEAL, lw=0.9, ls="--")
ax.text(c1 - 0.4, 92.5, f"꺾이는 온도 {c1:.1f}°C", color=TEAL, fontsize=8, ha="right")
ax.text(c2 + 0.4, 92.5, f"꺾이는 온도 {c2:.1f}°C", color=TEAL, fontsize=8, ha="left")
ax.annotate(f"1도 추워질 때마다\n{b1:.2f}GW 증가", (2, float(bent([2])[0])), (-8.5, 60.5), fontsize=8, color=NAVY, bbox={"fc": "white", "ec": "none", "alpha": 0.85, "pad": 1.5}, arrowprops={"arrowstyle": "->", "color": NAVY, "lw": 0.8})
ax.annotate(f"1도 더워질 때마다\n{b2:.2f}GW 증가", (26, float(bent([26])[0])), (23.2, 57.0), fontsize=8, color=NAVY, bbox={"fc": "white", "ec": "none", "alpha": 0.85, "pad": 1.5}, arrowprops={"arrowstyle": "->", "color": NAVY, "lw": 0.8})
ax.text((c1 + c2) / 2, 86.5, f"두 온도 사이에서는\n기온과 관계없이\n{b0:.1f}GW", color=NAVY, fontsize=8, ha="center", bbox={"fc": "white", "ec": "none", "alpha": 0.85, "pad": 1.5})
ax.set_ylim(55, 95)
ax.set_xlabel("하루 평균 기온(°C)", fontsize=8)
ax.set_ylabel("하루 평균 전력수요(GW)", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.12))
frame(ax)
fig.tight_layout()
fig.savefig("fig_1_7.png", dpi=300)
plt.show()

## 1.5 시험 자료를 고르는 기준

In [ ]:
def unseen(make, known, parts=5, seed=0):
    """known을 다섯 묶음으로 나눠, 한 묶음씩 가려 두고 나머지로 만든 식에게 가린 묶음을 묻는다."""
    order = np.random.default_rng(seed).permutation(len(known))
    sq = []
    for part in np.array_split(order, parts):
        held = known.iloc[part]
        f = make(known.drop(held.index))
        sq.append((held["GW"].to_numpy() - f(held["기온"])) ** 2)
    return round(float(np.sqrt(np.concatenate(sq).mean())), 2)

MAKERS = {
    "이웃 1일": lambda known: (lambda temps: neighbors(temps, 1, known)),
    "이웃 12일": lambda known: (lambda temps: neighbors(temps, 12, known)),
    "4차식": lambda known: polynomial(4, known),
    "꺾은 직선": hinge,
}
rows = []
for name, make in MAKERS.items():
    f = make(train)
    rows.append((name, rmse(train["GW"], f(train["기온"])), unseen(make, train), rmse(test["GW"], f(test["기온"]))))
layers = pd.DataFrame(rows, columns=["식", "본 날", "같은 해의 못 본 날", "다른 해"]).set_index("식")
print(layers.to_string())

In [ ]:
tries = [unseen(hinge, train, seed=s) for s in range(10)]
print(tries)
print(min(tries), max(tries))

In [ ]:
miss = test["GW"] - bent(test["기온"])
print(round(float(miss.mean()), 2), round(float((train["GW"] - bent(train["기온"])).mean()), 2) + 0.0)
print(miss.groupby(miss.index.month).mean().round(2).to_string())

In [ ]:
# 그림 1-9
by_month = miss.groupby(miss.index.month).mean()
fig, ax = plt.subplots(figsize=(7, 3.4))
face = [to_rgba(BRICK, 0.55) if v > 0 else to_rgba(NAVY, 0.22) for v in by_month]
edge = [BRICK if v > 0 else NAVY for v in by_month]
ax.bar(by_month.index, by_month, color=face, edgecolor=edge, lw=0.9, width=0.62)
for mth, v in by_month.items():
    ax.text(mth, v + (0.12 if v > 0 else -0.12), f"{v:+.2f}", ha="center", va="bottom" if v > 0 else "top", fontsize=7.5)
ax.axhline(0, color=GRAY, lw=0.8)
ax.axhline(miss.mean(), color=TEAL, lw=1.0, ls="--")
ax.text(12.7, miss.mean() + 0.15, f"한 해 평균\n{miss.mean():+.2f}GW", color=TEAL, fontsize=8, ha="left", va="bottom")
ax.set_xlim(0.2, 14.3)
ax.set_xticks(range(1, 13), [f"{m}월" for m in range(1, 13)])
ax.axvspan(1.5, 6.5, color=NAVY, alpha=0.05)
ax.axvspan(6.5, 9.5, color=BRICK, alpha=0.06)
ax.text(4, 2.75, "2~6월: 식의 답보다 2GW 이상 적게 썼다", color=NAVY, fontsize=8, ha="center")
ax.text(8, -4.1, "7~9월: 식의 답보다\n많이 썼다", color=BRICK, fontsize=8, ha="center")
ax.set_ylim(-4.6, 3.2)
ax.set_ylabel("실제 - 식의 답(GW)", fontsize=8)
ax.set_xlabel("2025년의 달", fontsize=8)
frame(ax)
fig.tight_layout()
fig.savefig("fig_1_9.png", dpi=300)
plt.show()

In [ ]:
print(rmse(test["GW"], bent(test["기온"]) + miss.mean()))